# Solution - Exercise: Recent Development, Large Language Models (LLMs)

## Task 1

### GPT-4o (OpenAI, 2024)
Multimodal model (text / audio / image / video) with real-time API capabilities around 300 ms voice latency.
- Advantage: High speed plus broad modality support for production-ready chat.
- Disadvantage: Closed-source, relatively high API costs.

### Gemini 2.5 Pro (Google DeepMind, 2025)
Natively multimodal with a context window up to 1 million tokens and tunable thinking time, excelling at coding and complex reasoning.
- Advantage: Huge context + adaptive reasoning deliver state-of-the-art math/code results.
- Disadvantage: Closed-source, latency through reasoning time, early reviewers notice occasional accuracy hiccups.

### Qwen 2.5 (Alibaba, 2024/2025)
Open-weight dense model family (0.5 B to 72 B parameters) trained on up to 18 T tokens; the API-only sibling Qwen 2.5-Max (a MoE trained on over 20 T tokens) surpasses DeepSeek-V3 on multiple benchmarks.
- Advantage: Top-tier open model family with permissive licensing (Apache 2.0 for most sizes) and competitive benchmark scores.
- Disadvantage: The strongest variant (Qwen 2.5-Max) is closed and API-only; the larger open models demand substantial GPU power, curbing easy local use.

### DeepSeek-V3 (DeepSeek AI, 2025)
671 B-parameter MoE (37 B active) trained on 14.8 T tokens; the weights are openly released.
- Advantage: Very good performance on benchmarks like MMLU-Pro while MoE sparsity keeps active parameters modest.
- Disadvantage: Total size (671 B) still strains hardware and storage for self-hosting despite sparse inference.

## Programming Setup
Install dependencies and configure API keys.

In [1]:
%pip install -qU google-genai datasets transformers sentence_transformers accelerate bitsandbytes langchain_text_splitters


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: /home/andre/git/LLMs-Lecture/exercise/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass

try:
  from google.colab import userdata
  os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception as e:
  pass

if not os.environ.get("GEMINI_API_KEY"):
  os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

Enter API key for Google Gemini:  ········


## Task 2 - Practical Benchmarking

In [3]:
from datasets import load_dataset
from abc import ABC, abstractmethod
import signal
from contextlib import contextmanager, redirect_stdout
import gc
import io
import os
import logging
import warnings
from google import genai
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# bitsandbytes emits harmless FutureWarnings and a kernel-fallback notice on CPU-only machines
warnings.filterwarnings("ignore", category=FutureWarning, module="bitsandbytes")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

class LLM(ABC):
    @abstractmethod
    def __init__(self, model_name: str = "<Missing Model Name>", **model_args) -> None:
        self.model_name = model_name
        pass

    @abstractmethod
    def prompt(self, message: str) -> str:
        pass

class TransformersLLM(LLM):
    def __init__(self, model_name: str, **model_args) -> None:
        super().__init__(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", **model_args)

    def prompt(self, message: str) -> str:
        chatPrompt = [{"role": "user", "content": message}]

        formattedChat = self.tokenizer.apply_chat_template(chatPrompt, tokenize=False, add_generation_prompt=True)
        # The chat template already inserted all special tokens
        inputs = self.tokenizer(formattedChat, return_tensors="pt", add_special_tokens=False).to(self.model.device)
        maximalNewTokens = 1024
        skipSpecialTokens = True
        output = self.tokenizer.decode(self.model.generate(**inputs, max_new_tokens = maximalNewTokens)[0][inputs["input_ids"].size(1):], skip_special_tokens = skipSpecialTokens)
        return output

class GeminiLLM(LLM):
    def __init__(self, model_name: str, **model_args) -> None:
        super().__init__(model_name, **model_args)
        self.client = genai.Client()  # reads GEMINI_API_KEY from the environment
        self.api_model_name = model_name

    def prompt(self, message : str) -> str:
        try:
            output = self.client.models.generate_content(model=self.api_model_name, contents=message)
            return output.text or ""
        except Exception as e:
            print(f"{self.model_name}: request failed ({e})")
            return ""

class HumanEval:
    INSTRUCTION = "Complete the following Python function. Return the full function definition in a single ```python code block.\n\n"

    def __init__(self, number_of_tasks: int = 5) -> None:
        self.tasks = load_dataset("openai/openai_humaneval", split=f"test[:{number_of_tasks}]")
        self.number_of_tasks = number_of_tasks

    @staticmethod
    @contextmanager
    def time_limit(seconds: int):
        def handler(signum, frame):
            raise TimeoutError("Timed out!")
        old = signal.signal(signal.SIGALRM, handler)
        signal.alarm(seconds)
        try:
            yield
        finally:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, old)

    @staticmethod
    def run_task(task, code_str: str, timeout_s: int = 5) -> bool:
        """
        Execute model code and run the task's unit tests.
        Returns True if all asserts pass, False otherwise.
        """
        ns = {}
        try:
            # Swallow print output of the generated code so it does not clutter the score list
            with HumanEval.time_limit(timeout_s), redirect_stdout(io.StringIO()):
                # 1) Exec model code
                exec(code_str, ns)

                # 2) Exec test code (defines check)
                exec(task["test"], ns)

                # 3) Call check with the candidate function
                entry_point = task["entry_point"]
                candidate_fn = ns[entry_point]
                ns["check"](candidate_fn)   # raises AssertionError if fail
            return True
        except Exception as e:
            # For debugging you could log e or print(task["task_id"], e)
            return False

    def getCode(self, gen: str) -> str:
        # Parse the answer from the response; the language tag is optional
        pattern = re.compile(r'```(?:python)?\s(.*?)```', re.DOTALL)
        matches = pattern.findall(gen)

        code = gen
        # Check if any match was found
        if matches:
            code = matches[0]

        return code

    def evaluate(self, *llms: LLM, timeout_s: int = 5) -> dict[str, float]:
        scores = {}
        print("Score\n------")
        for llm in llms:
            passed = 0
            for task in self.tasks:
                gen = llm.prompt(self.INSTRUCTION + task["prompt"])
                code = self.getCode(gen)
                if self.run_task(task, code, timeout_s):
                    passed += 1
            score = passed / self.number_of_tasks
            scores[llm.model_name] = score
            print(f"{llm.model_name}: {score:.3f}")
        return scores

# --- Load models ------------------------------------------------------------
closed_model_name = "gemini-2.5-flash"
closed_model = GeminiLLM(model_name=closed_model_name)

open_model_name = "Qwen/Qwen2.5-3B-Instruct"
open_model   = TransformersLLM(model_name=open_model_name)

# --- Build 5-item HumanEval subset -----------------------------------------
human_eval = HumanEval(5)

# --- Run evaluation ---------------------------------------------------------
scores = human_eval.evaluate(closed_model, open_model)

# --- Clear cuda memory ------------------------------------------------------
del open_model
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Score
------
gemini-2.5-flash: 0.800
Qwen/Qwen2.5-3B-Instruct: 0.800


## Task 3 - Quantisation (4-bit Qwen 2.5 3B)

In [4]:
from transformers import BitsAndBytesConfig
import gc

# --- Set configuration ------------------------------------------------------
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)

# --- Load models ------------------------------------------------------------
open_model_quantized = TransformersLLM(open_model_name, quantization_config=quantization_config)

# --- Run evaluation ---------------------------------------------------------
scores = human_eval.evaluate(open_model_quantized)

del open_model_quantized
gc.collect()
torch.cuda.empty_cache()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 0 files: 0it [00:00, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Score
------
Qwen/Qwen2.5-3B-Instruct: 0.800


## Task 4 - Prompting Techniques

In [5]:
class PromptingTechniqueLLM(GeminiLLM):
    def __init__(self, model_name: str, **model_args) -> None:
        super().__init__(model_name, **{})
        self.pre_message = model_args["pre_message"]
        self.model_name += model_args["post_name"]

    def prompt(self, message: str) -> str:
        return super().prompt(self.pre_message + message)

# --- Set configuration ------------------------------------------------------
cot_prompt = "Think step-by-step.\n"
role_prompt = "You are a senior competitive programmer.\n"

# --- Load models ------------------------------------------------------------
direct = PromptingTechniqueLLM(closed_model_name, pre_message="", post_name=" (Direct)")
cot    = PromptingTechniqueLLM(closed_model_name, pre_message=cot_prompt, post_name=" (CoT)")
role   = PromptingTechniqueLLM(closed_model_name, pre_message=role_prompt, post_name=" (Role)")

# --- Run evaluation ---------------------------------------------------------
scores = human_eval.evaluate(direct, cot, role)

Score
------
gemini-2.5-flash (Direct): 1.000
gemini-2.5-flash (CoT): 1.000
gemini-2.5-flash (Role): 1.000


## Task 5 - Retrieval-Augmented Prototype

In [1]:
import re
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder, util

# ------------------------------------------------------------------- #
#     Load corpus and build the index once                            #
# ------------------------------------------------------------------- #
SOURCE_DIR = Path("RAG")
SOURCE_FILES = ["Bavaria.md", "Munich.md", "Roman_Empire.md"]

def clean(md : str) -> str:
    # The markdown comes from a PDF conversion: drop image placeholders and
    # flatten HTML tables into plain text lines so the facts inside embed well
    md = md.replace("<!-- image -->", "")
    md = re.sub(r"</tr>\s*<tr[^>]*>", "\n", md)
    md = re.sub(r"</?(?:table|tr)[^>]*>", "\n", md)
    md = re.sub(r"</td>\s*<td[^>]*>", ": ", md)
    md = re.sub(r"</?td[^>]*>", " ", md)
    md = re.sub(r"\n{3,}", "\n\n", md)
    return md

text = ""
for fp in SOURCE_FILES:
    path = SOURCE_DIR / fp
    if not path.is_file():
        raise FileNotFoundError(f"Missing source file: {path}")
    with open(path, "r") as text_file:
            text += clean(text_file.read()) + "\n"

# Split the text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 3000, chunk_overlap = 200)
passages = text_splitter.split_text(text)

topK = 32                          # Number of passages we want to retrieve with the bi-encoder
biEncoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
biEncoder.max_seq_length = 512
corpusEmbeddings = biEncoder.encode(passages, convert_to_tensor = True, show_progress_bar = True)

crossEncoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

def rag(message : str):
    ##### Semantic Search #####

    # Encode the query using the bi-encoder and find potentially relevant passages
    questionEmbedding = biEncoder.encode(message, convert_to_tensor = True)
    questionEmbedding = questionEmbedding.cpu()
    hits = util.semantic_search(questionEmbedding, corpusEmbeddings, top_k = topK)
    hits = hits[0]

    ##### Re-Ranking #####
    # Now, score all retrieved passages with the cross_encoder
    crossInp = [[message, passages[hit["corpus_id"]]] for hit in hits]
    crossScores = crossEncoder.predict(crossInp)

    # Sort results by the cross-encoder scores
    for idx in range(len(crossScores)):
        hits[idx]["cross-score"] = crossScores[idx]

    # Output of top-10 hits from re-ranker
    hits = sorted(hits, key = lambda x: x["cross-score"], reverse = True)
    contexts = ""
    for hit in hits[0 : 10]:
        contexts = contexts + passages[hit["corpus_id"]] + "\n\n"

    return contexts

# ------------------------------------------------------------------- #
#     Simple chat loop                                                #
# ------------------------------------------------------------------- #
questions = [
    "How many inhabitants does Bavaria have?",
    "Who is the current Minister-President of Bavaria and which is his party?",
    "How many inhabitants does Munich have?",
    "What is the area of Munich?",
    "Explain the term Pax Romana.",
]

for q in questions:
    print(closed_model.prompt(f"{q} Use the following information:\n{rag(q)}"))

ModuleNotFoundError: No module named 'langchain_text_splitters'

# Additional Tasks

The following tasks accompany the lecture sections *Reasoning Models* and *Agents and Tool Calling*.

## Task 6 - Reasoning and Test-Time Compute

Reasoning models generate an extended chain of thought ("thinking") before answering, trading additional inference compute for higher accuracy. Gemini 2.5 Flash exposes this trade-off directly through a thinking budget. Compare the model with thinking disabled (`thinking_budget=0`) against dynamic thinking (`thinking_budget=-1`) on a set of trick questions: measure accuracy, thinking tokens and latency. Observe the trade-off from the lecture: thinking costs extra tokens and time, and simple questions can trigger unnecessarily long reasoning chains ("overthinking").

In [2]:
import time
from google.genai import types

client = closed_model.client

# --- Trick questions with verifiable numeric answers -------------------------
puzzles = [
    ("Alice has 4 sisters and 1 brother. How many sisters does Alice's brother have?", 5),
    ("What is 347 * 268?", 92996),
    ("I have 3 apples today. Yesterday I ate 2 apples. How many apples do I have today?", 3),
    ("A farmer has 17 sheep. All but 9 run away. How many sheep are left?", 9),
    ("What is the sum of all integers from 1 to 200 that are divisible by 3?", 6633),
]

def ask(question: str, thinking_budget: int):
    """Query Gemini with a fixed thinking budget; 0 disables thinking, -1 lets the model decide."""
    config = types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=thinking_budget))
    start = time.time()
    response = client.models.generate_content(model=closed_model_name,
                                              contents=question, config=config)
    seconds = time.time() - start
    thoughtTokens = response.usage_metadata.thoughts_token_count or 0
    return response.text or "", seconds, thoughtTokens

# --- Compare no thinking vs. dynamic thinking ---------------------------------
suffix = "\n\nEnd your reply with only the final number."
for label, budget in [("Thinking off  (budget=0) ", 0), ("Dynamic thinking (budget=-1)", -1)]:
    correct, thoughtTokens, seconds = 0, 0, 0.0
    for question, answer in puzzles:
        text, s, t = ask(question + suffix, budget)
        numbers = re.findall(r"-?\d+", text.replace(",", ""))
        correct += bool(numbers) and int(numbers[-1]) == answer
        thoughtTokens += t
        seconds += s
    print(f"{label}: accuracy {correct}/{len(puzzles)}, "
          f"thinking tokens {thoughtTokens}, time {seconds:.1f} s")

ModuleNotFoundError: No module named 'google'

## Task 7 - Tool Calling

Tool calling lets an LLM interact with external systems: given the schema of a tool, the model emits a structured call (tool name plus JSON arguments) instead of plain text; the application executes the call and feeds the result back. Reproduce the weather example from the lecture with a mock `get_weather` implementation and inspect each step of the loop. The model never executes anything itself; the application stays in control.

In [8]:
from google.genai import types

client = closed_model.client

# --- 1) Register the tool: a schema plus a mock implementation ---------------
get_weather_declaration = {
    "name": "get_weather",
    "description": "Returns the current weather condition for a given city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "Name of the city"},
        },
        "required": ["city"],
    },
}

def get_weather(city: str) -> dict:
    return {"condition": "rain"}   # mock; a real application would call a weather API

tool_config = types.GenerateContentConfig(
    tools=[types.Tool(function_declarations=[get_weather_declaration])])

# --- 2) The model emits a structured call instead of an answer ---------------
question = "Do I need an umbrella in Munich today?"
contents = [types.Content(role="user", parts=[types.Part(text=question)])]
response = client.models.generate_content(model=closed_model_name, contents=contents, config=tool_config)

call = next(part.function_call for part in response.candidates[0].content.parts if part.function_call)
print(f"Tool call: {call.name}({dict(call.args)})")

# --- 3) The application executes the call, not the model ---------------------
result = get_weather(**dict(call.args))
print(f"Tool result: {result}")

# --- 4) Feed the result back; the model generates the final answer -----------
contents.append(response.candidates[0].content)
contents.append(types.Content(role="user",
                parts=[types.Part.from_function_response(name=call.name, response=result)]))
final = client.models.generate_content(model=closed_model_name, contents=contents, config=tool_config)
print(f"Final answer: {final.text}")

Tool call: get_weather({'city': 'Munich'})
Tool result: {'condition': 'rain'}
Final answer: Yes, you will need an umbrella in Munich today as it is raining.


## Task 8 - Agentic RAG (Mini-Agent)

An agent pursues a goal over multiple steps in a loop: it plans, calls tools, observes the results and adapts until the task is solved. Build a minimal agent that answers multi-hop questions over the Task 5 corpus with two tools: `search_corpus` (the retriever from Task 5) and `calculator`. In contrast to Task 5, the model itself decides when and what to retrieve (agentic RAG). Note the `max_steps` guard: errors compound over long task horizons, and the allowlist in `calculator` keeps the agent from executing arbitrary code.

Requires the cells of Task 2 (client) and Task 5 (retriever).

In [9]:
from google.genai import types

client = closed_model.client

# --- Tools: retrieval over the Task 5 corpus and a calculator ----------------
search_corpus_declaration = {
    "name": "search_corpus",
    "description": "Searches the document corpus (Bavaria, Munich, Roman Empire) and returns the most relevant passages.",
    "parameters": {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "Search query"}},
        "required": ["query"],
    },
}

calculator_declaration = {
    "name": "calculator",
    "description": "Evaluates a basic arithmetic expression, e.g. '1512491 / 310.7'.",
    "parameters": {
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "Arithmetic expression"}},
        "required": ["expression"],
    },
}

def calculator(expression: str) -> str:
    # Allowlist so the agent cannot make us execute arbitrary code
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "Error: only basic arithmetic is allowed."
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

TOOL_FUNCTIONS = {
    "search_corpus": lambda query: rag(query),   # the retriever from Task 5
    "calculator": calculator,
}

agent_config = types.GenerateContentConfig(
    tools=[types.Tool(function_declarations=[search_corpus_declaration, calculator_declaration])],
    system_instruction="Solve the task step by step: look up facts with search_corpus and do all arithmetic with calculator.")

# --- The agent loop: plan, call tools, observe, adapt ------------------------
def run_agent(question: str, max_steps: int = 8) -> str:
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]
    for step in range(max_steps):
        response = client.models.generate_content(model=closed_model_name, contents=contents, config=agent_config)
        parts = response.candidates[0].content.parts or []
        calls = [part.function_call for part in parts if part.function_call]
        if not calls:                      # no tool call: the answer is final
            return response.text

        contents.append(response.candidates[0].content)
        for call in calls:
            result = TOOL_FUNCTIONS[call.name](**dict(call.args))
            print(f"  Step {step + 1}: {call.name}({dict(call.args)}) -> {str(result)[:100]!r}")
            contents.append(types.Content(role="user",
                            parts=[types.Part.from_function_response(name=call.name, response={"result": str(result)})]))
    return "Agent stopped: maximum number of steps reached."

print(run_agent("What is the population density of Munich in inhabitants per square kilometre? "
                "Look up the population and the area, then calculate."))

  Step 1: search_corpus({'query': 'population of Munich'}) -> 'Historical population: Year: Pop.: ±%: 1500: 13,447: —: 1600: 21,943: +63.2%: 1750: 32,000: +45.8%: '
  Step 2: calculator({'expression': '1604384 / 310.71'}) -> '5163.605934794503'
The population density of Munich is approximately 5163.61 inhabitants per square kilometre.
